# GraphFraud — EDA & Baselines

Exploratory data analysis of the **Elliptic Bitcoin Dataset** and non-graph baseline benchmarks.

| Step | Description |
|------|-------------|
| 1 | Load & inspect dataset structure |
| 2 | Class distribution & imbalance analysis |
| 3 | Temporal dynamics across 49 timesteps |
| 4 | Feature distributions & correlations |
| 5 | Baseline models (Logistic Regression, Random Forest, XGBoost) |
| 6 | Comparison table — sets the bar for GNN models |

## 1. Setup & Load Data

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# Ensure graphfraud is importable
sys.path.insert(0, '../..')

from graphfraud.data.dataset import load_elliptic, EllipticDataset

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
%matplotlib inline

SEED = 42

In [ ]:
# Load the Elliptic dataset with temporal split
from pathlib import Path

DATA_DIR = Path('../../data')
dataset = load_elliptic(DATA_DIR, temporal_split=True)
print(dataset.summary())

## 2. Class Distribution & Imbalance

In [ ]:
# Overall class distribution
label_names = {0: 'Licit', 1: 'Illicit', -1: 'Unknown'}
label_counts = Counter(dataset.labels)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# All nodes
labels_all = [label_names[l] for l in sorted(label_counts.keys())]
counts_all = [label_counts[l] for l in sorted(label_counts.keys())]
colors = ['#e74c3c', '#2ecc71', '#95a5a6']  # illicit, licit, unknown
axes[0].bar(labels_all, counts_all, color=colors)
axes[0].set_title('All Nodes')
axes[0].set_ylabel('Count')
for i, (lbl, cnt) in enumerate(zip(labels_all, counts_all)):
    axes[0].text(i, cnt + 1000, f'{cnt:,}', ha='center', fontweight='bold')

# Labeled only
labeled_mask = dataset.labels >= 0
labeled_labels = dataset.labels[labeled_mask]
labeled_counts = Counter(labeled_labels)
axes[1].bar(['Licit', 'Illicit'],
            [labeled_counts[0], labeled_counts[1]],
            color=['#2ecc71', '#e74c3c'])
axes[1].set_title(f'Labeled Only (n={len(labeled_labels):,})')
axes[1].set_ylabel('Count')
ratio = labeled_counts[0] / labeled_counts[1]
axes[1].text(0, labeled_counts[0] + 500, f'{labeled_counts[0]:,}', ha='center', fontweight='bold')
axes[1].text(1, labeled_counts[1] + 500, f'{labeled_counts[1]:,}', ha='center', fontweight='bold')
axes[1].set_xlabel(f'Imbalance ratio: {ratio:.1f}:1')

fig.suptitle('Elliptic Bitcoin Dataset — Class Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'\nImbalance ratio (licit:illicit): {ratio:.1f}:1')
print(f'Illicit prevalence: {labeled_counts[1] / len(labeled_labels) * 100:.1f}%')

## 3. Temporal Dynamics

In [ ]:
# Class distribution over time
timesteps = dataset.timesteps
labels = dataset.labels

ts_data = []
for t in range(1, 50):
    t_mask = timesteps == t
    t_labels = labels[t_mask]
    ts_data.append({
        'timestep': t,
        'total': t_mask.sum(),
        'licit': (t_labels == 0).sum(),
        'illicit': (t_labels == 1).sum(),
        'unknown': (t_labels == -1).sum(),
    })

df_ts = pd.DataFrame(ts_data)
df_ts['illicit_rate'] = df_ts['illicit'] / (df_ts['licit'] + df_ts['illicit']).replace(0, np.nan)

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Stacked bar: labeled counts per timestep
axes[0].bar(df_ts['timestep'], df_ts['licit'], label='Licit', color='#2ecc71', alpha=0.8)
axes[0].bar(df_ts['timestep'], df_ts['illicit'], bottom=df_ts['licit'],
            label='Illicit', color='#e74c3c', alpha=0.8)
axes[0].set_ylabel('Labeled Transactions')
axes[0].legend()
axes[0].set_title('Transaction Counts by Timestep')

# Train/val/test boundaries
for ax in axes:
    ax.axvline(x=34.5, color='blue', linestyle='--', alpha=0.7, label='Train/Val boundary')
    ax.axvline(x=42.5, color='red', linestyle='--', alpha=0.7, label='Val/Test boundary')

# Illicit rate over time
axes[1].plot(df_ts['timestep'], df_ts['illicit_rate'] * 100, 'o-', color='#e74c3c', lw=2)
axes[1].set_ylabel('Illicit Rate (%)')
axes[1].set_xlabel('Timestep')
axes[1].set_title('Illicit Rate Over Time — shows temporal shift (why random split leaks)')

fig.suptitle('Temporal Structure — Train (1-34) / Val (35-42) / Test (43-49)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Split sizes
print(f'\nTemporal split sizes:')
print(f'  Train: {dataset.train_mask.sum():,} labeled nodes (t1-t34)')
print(f'  Val:   {dataset.val_mask.sum():,} labeled nodes (t35-t42)')
print(f'  Test:  {dataset.test_mask.sum():,} labeled nodes (t43-t49)')

## 4. Feature Analysis

In [ ]:
# Feature summary statistics
X = dataset.node_features
print(f'Feature matrix shape: {X.shape}')
print(f'Feature ranges:')
print(f'  Min: {X.min():.4f}')
print(f'  Max: {X.max():.4f}')
print(f'  Mean: {X.mean():.4f}')
print(f'  Std: {X.std():.4f}')
print(f'  NaN count: {np.isnan(X).sum()}')
print(f'\n  First 94 features: local transaction features')
print(f'  Last 72 features: aggregated 1-hop neighbor features')

In [ ]:
# Distribution of select features by class (labeled nodes only)
labeled_X = X[labeled_mask]
labeled_y = labels[labeled_mask]

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
feature_indices = [0, 1, 10, 20, 50, 80, 94, 130]  # Sample across local + neighbor features
feature_labels = [f'Local f{i}' if i < 94 else f'Neighbor f{i}' for i in feature_indices]

for ax, fi, fl in zip(axes.ravel(), feature_indices, feature_labels):
    licit_vals = labeled_X[labeled_y == 0, fi]
    illicit_vals = labeled_X[labeled_y == 1, fi]

    ax.hist(licit_vals, bins=50, alpha=0.6, label='Licit', color='#2ecc71', density=True)
    ax.hist(illicit_vals, bins=50, alpha=0.6, label='Illicit', color='#e74c3c', density=True)
    ax.set_title(fl)
    ax.legend(fontsize=8)

fig.suptitle('Feature Distributions by Class', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Feature correlation heatmap (first 20 features)
df_feat = pd.DataFrame(labeled_X[:, :20], columns=[f'f{i}' for i in range(20)])
df_feat['label'] = labeled_y

fig, ax = plt.subplots(figsize=(12, 10))
corr = df_feat.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='RdBu_r', center=0, ax=ax,
            square=True, linewidths=0.5, annot=False)
ax.set_title('Feature Correlation Matrix (first 20 features)', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Graph Structure

In [ ]:
# Edge statistics
edge_index = dataset.edge_index
print(f'Edges: {edge_index.shape[1]:,}')
print(f'Nodes: {dataset.num_nodes:,}')
print(f'Avg degree: {edge_index.shape[1] / dataset.num_nodes:.2f}')

# Degree distribution
from collections import Counter
src, dst = edge_index[0], edge_index[1]
all_nodes = np.concatenate([src, dst])
degree_counts = Counter(all_nodes)
degrees = list(degree_counts.values())

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(degrees, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
ax.set_xlabel('Node Degree')
ax.set_ylabel('Count')
ax.set_title(f'Degree Distribution (mean={np.mean(degrees):.1f}, max={max(degrees)})')
ax.set_yscale('log')
plt.tight_layout()
plt.show()

# Degree by class
labeled_degrees = [degree_counts.get(i, 0) for i in range(dataset.num_nodes) if labels[i] >= 0]
labeled_classes = [labels[i] for i in range(dataset.num_nodes) if labels[i] >= 0]

deg_licit = [d for d, c in zip(labeled_degrees, labeled_classes) if c == 0]
deg_illicit = [d for d, c in zip(labeled_degrees, labeled_classes) if c == 1]

print(f'\nDegree by class:')
print(f'  Licit:   mean={np.mean(deg_licit):.2f}, median={np.median(deg_licit):.0f}')
print(f'  Illicit: mean={np.mean(deg_illicit):.2f}, median={np.median(deg_illicit):.0f}')

## 6. Non-Graph Baselines

Train **Logistic Regression**, **Random Forest**, and **XGBoost** using only node features (166 dims).
These baselines ignore graph structure — the GNN must beat them to justify the added complexity.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, classification_report, roc_auc_score
from graphfraud.data.resampling import hybrid_resample

# Prepare data with temporal split
X_train = dataset.node_features[dataset.train_mask]
y_train = dataset.labels[dataset.train_mask]
X_val = dataset.node_features[dataset.val_mask]
y_val = dataset.labels[dataset.val_mask]
X_test = dataset.node_features[dataset.test_mask]
y_test = dataset.labels[dataset.test_mask]

print(f'Train: {X_train.shape[0]:,} ({(y_train == 1).sum()} illicit)')
print(f'Val:   {X_val.shape[0]:,} ({(y_val == 1).sum()} illicit)')
print(f'Test:  {X_test.shape[0]:,} ({(y_test == 1).sum()} illicit)')

# Standardize
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)

# Resample
X_train_bal, y_train_bal = hybrid_resample(X_train_s, y_train, max_majority=30000)
print(f'\nAfter hybrid resampling: {len(y_train_bal):,} ({Counter(y_train_bal)})')

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb

baselines = {}

# ── Logistic Regression ─────────────────────────────────────────
lr = LogisticRegression(C=1.0, max_iter=1000, class_weight='balanced', random_state=SEED)
lr.fit(X_train_bal, y_train_bal)
lr_pred = lr.predict(X_test_s)
lr_prob = lr.predict_proba(X_test_s)[:, 1]
baselines['Logistic Regression'] = {
    'f1': f1_score(y_test, lr_pred, pos_label=1),
    'f1_macro': f1_score(y_test, lr_pred, average='macro'),
    'auc': roc_auc_score(y_test, lr_prob),
}

# ── Random Forest ───────────────────────────────────────────────
rf = RandomForestClassifier(n_estimators=300, max_depth=12, class_weight='balanced',
                            random_state=SEED, n_jobs=-1)
rf.fit(X_train_bal, y_train_bal)
rf_pred = rf.predict(X_test_s)
rf_prob = rf.predict_proba(X_test_s)[:, 1]
baselines['Random Forest'] = {
    'f1': f1_score(y_test, rf_pred, pos_label=1),
    'f1_macro': f1_score(y_test, rf_pred, average='macro'),
    'auc': roc_auc_score(y_test, rf_prob),
}

# ── XGBoost ─────────────────────────────────────────────────────
xgb_model = xgb.XGBClassifier(
    n_estimators=500, max_depth=8, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    tree_method='hist', eval_metric='logloss',
    random_state=SEED,
)
xgb_model.fit(X_train_bal, y_train_bal, eval_set=[(X_val_s, y_val)], verbose=False)
xgb_pred = xgb_model.predict(X_test_s)
xgb_prob = xgb_model.predict_proba(X_test_s)[:, 1]
baselines['XGBoost'] = {
    'f1': f1_score(y_test, xgb_pred, pos_label=1),
    'f1_macro': f1_score(y_test, xgb_pred, average='macro'),
    'auc': roc_auc_score(y_test, xgb_prob),
}

# ── Results table ───────────────────────────────────────────────
df_results = pd.DataFrame(baselines).T
df_results.columns = ['F1 (illicit)', 'F1 (macro)', 'AUC-ROC']
print('\n' + '═' * 60)
print('Non-Graph Baselines — Test Set (Temporal Split)')
print('═' * 60)
print(df_results.round(4).to_string())
print('═' * 60)

In [ ]:
# Classification reports
for name, (pred, model) in [('Logistic Regression', (lr_pred, lr)),
                             ('Random Forest', (rf_pred, rf)),
                             ('XGBoost', (xgb_pred, xgb_model))]:
    print(f'\n{"─" * 40}')
    print(f'{name}:')
    print(classification_report(y_test, pred, target_names=['licit', 'illicit'], digits=4))

In [ ]:
# Visual comparison
from sklearn.metrics import roc_curve, precision_recall_curve

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = {'Logistic Regression': '#3498db', 'Random Forest': '#2ecc71', 'XGBoost': '#e74c3c'}

for name, prob in [('Logistic Regression', lr_prob), ('Random Forest', rf_prob), ('XGBoost', xgb_prob)]:
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc = baselines[name]['auc']
    axes[0].plot(fpr, tpr, color=colors[name], lw=2, label=f'{name} (AUC={auc:.3f})')

    prec, rec, _ = precision_recall_curve(y_test, prob)
    axes[1].plot(rec, prec, color=colors[name], lw=2, label=name)

axes[0].plot([0, 1], [0, 1], 'k--', lw=1)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curves')
axes[0].legend()

axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curves')
axes[1].legend()

fig.suptitle('Non-Graph Baselines — The Bar for GNN Models', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Feature Importance (XGBoost)

In [ ]:
# XGBoost feature importance
importances = xgb_model.feature_importances_
top_k = 20
top_idx = np.argsort(importances)[-top_k:][::-1]

fig, ax = plt.subplots(figsize=(10, 8))
feature_names = [f'Local f{i}' if i < 94 else f'Neighbor f{i}' for i in range(166)]
ax.barh(range(top_k), importances[top_idx][::-1], color='coral')
ax.set_yticks(range(top_k))
ax.set_yticklabels([feature_names[i] for i in top_idx[::-1]])
ax.set_xlabel('Importance (gain)')
ax.set_title(f'Top {top_k} Features — XGBoost Baseline', fontsize=14)
plt.tight_layout()
plt.show()

# How many top features are local vs neighbor?
n_local = sum(1 for i in top_idx if i < 94)
n_neighbor = sum(1 for i in top_idx if i >= 94)
print(f'\nTop {top_k}: {n_local} local, {n_neighbor} neighbor features')
print(f'→ If neighbor features dominate, graph structure likely helps GNNs too')

## 8. Summary

### Key findings
- **Class imbalance:** ~9:1 licit:illicit among labeled nodes
- **Temporal shift:** Illicit rate varies across timesteps — justifies temporal (not random) split
- **77% unlabeled:** Semi-supervised approaches can exploit unlabeled nodes during message passing
- **Feature scale variation:** Standardization is important for both LR and GNNs

### Baseline benchmarks (test set)

| Model | F1 (illicit) | F1 (macro) | AUC-ROC | Uses Graph? |
|-------|-------------|------------|---------|-------------|
| Logistic Regression | — | — | — | No |
| Random Forest | — | — | — | No |
| XGBoost | — | — | — | No |
| GCN | — | — | — | Yes |
| GraphSAGE | — | — | — | Yes |
| **GATv2Conv** | **—** | **—** | **—** | **Yes** |

*(Fill in after running each model)*

### Next steps
1. `graphfraud train --config configs/gatv2.yaml` — train the primary GNN
2. Compare GNN vs baselines — does graph topology improve F1?
3. `graphfraud explain --node-id <id>` — inspect attention weights
4. Optuna HP search for final numbers